First of all, we create and configure the Spark session using Apache Spark.

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [6]:
spark = SparkSession.builder \
    .appName("CleaningSales") \
    .master("local[8]") \
    .getOrCreate()

In [7]:
spark.sparkContext.setLogLevel("ERROR")

Once the dataset has been loaded, we proceed with its cleaning and preparation for the ETL process.

In [8]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("sep", ";") \
    .csv("../datasets/dataset_sales.csv", header=True, inferSchema=True)

First of all, we select the columns of interest and we rename them.

In [9]:
df = df.drop("Total Nacional")

In [10]:
df = df.withColumnRenamed("Comunidades y Ciudades Autónomas", "Region") \
    .withColumnRenamed("Provincias", "City") \
    .withColumnRenamed("Periodo", "Period") \
    .withColumnRenamed("Régimen y estado", "Regime_Condition")

Now we can see that some columns are not in the format we need, so we will transform them.

In [11]:
df.select("Region").distinct().orderBy("Region").show(truncate=False)

+------------------------------+
|Region                        |
+------------------------------+
|NULL                          |
|01 Andalucía                  |
|02 Aragón                     |
|03 Asturias, Principado de    |
|04 Balears, Illes             |
|05 Canarias                   |
|06 Cantabria                  |
|07 Castilla y León            |
|08 Castilla - La Mancha       |
|09 Cataluña                   |
|10 Comunitat Valenciana       |
|11 Extremadura                |
|12 Galicia                    |
|13 Madrid, Comunidad de       |
|14 Murcia, Región de          |
|15 Navarra, Comunidad Foral de|
|16 País Vasco                 |
|17 Rioja, La                  |
|18 Ceuta                      |
|19 Melilla                    |
+------------------------------+



In [12]:
df = df.withColumn("Region",F.trim(F.regexp_replace(F.col("Region"), r"^\d+\s+", "")))

In [13]:
df.select("Region").distinct().orderBy("Region").show(truncate=False)

+---------------------------+
|Region                     |
+---------------------------+
|NULL                       |
|Andalucía                  |
|Aragón                     |
|Asturias, Principado de    |
|Balears, Illes             |
|Canarias                   |
|Cantabria                  |
|Castilla - La Mancha       |
|Castilla y León            |
|Cataluña                   |
|Ceuta                      |
|Comunitat Valenciana       |
|Extremadura                |
|Galicia                    |
|Madrid, Comunidad de       |
|Melilla                    |
|Murcia, Región de          |
|Navarra, Comunidad Foral de|
|País Vasco                 |
|Rioja, La                  |
+---------------------------+



In [14]:
df = df.withColumn("Region",
    F.when(
        F.col("Region").contains(","),
        F.concat_ws(
            " ",
            F.trim(F.element_at(F.split(F.col("Region"), ","), 2)),
            F.trim(F.element_at(F.split(F.col("Region"), ","), 1))
        )
    ).otherwise(F.col("Region"))
)

In [15]:
df.select("Region").distinct().orderBy("Region").show(truncate=False)

+--------------------------+
|Region                    |
+--------------------------+
|NULL                      |
|Andalucía                 |
|Aragón                    |
|Canarias                  |
|Cantabria                 |
|Castilla - La Mancha      |
|Castilla y León           |
|Cataluña                  |
|Ceuta                     |
|Comunidad Foral de Navarra|
|Comunidad de Madrid       |
|Comunitat Valenciana      |
|Extremadura               |
|Galicia                   |
|Illes Balears             |
|La Rioja                  |
|Melilla                   |
|País Vasco                |
|Principado de Asturias    |
|Región de Murcia          |
+--------------------------+



In [16]:
df = df.dropna(subset=["Region", "City"])

In [17]:
df = df.withColumn("City",F.trim(F.regexp_replace(F.col("City"), r"^\d+\s+", "")))

In [18]:
df = df.withColumn("City",
    F.when(
        F.col("City").contains(","),
        F.concat_ws(
            " ",
            F.trim(F.element_at(F.split(F.col("City"), ","), 2)),
            F.trim(F.element_at(F.split(F.col("City"), ","), 1))
        )
    ).otherwise(F.col("City"))
)

Let's see the different values we can obtain in "Regime_Condition" column.

In [19]:
df.select("Regime_Condition").distinct().show(truncate=False)

+------------------+
|Regime_Condition  |
+------------------+
|Vivienda protegida|
|Vivienda libre    |
|Vivienda usada    |
|Vivienda nueva    |
|Viviendas: Total  |
+------------------+



In [20]:
df = df.withColumn(
    "Regime",
    F.when(F.col("Regime_Condition") == "Vivienda protegida", "Protegida")
     .when(F.col("Regime_Condition") == "Vivienda libre", "Libre")
)

In [21]:
df = df.withColumn(
    "Condition",
    F.when(F.col("Regime_Condition") == "Vivienda nueva", "Nueva")
     .when(F.col("Regime_Condition") == "Vivienda usada", "Segunda mano")
)

In [22]:
df = df.drop("Regime_Condition").dropna(subset=["Regime", "Condition"], how ="all")

In [23]:
df.show(truncate=False)

+---------+-------+-------+-----+------+---------+
|Region   |City   |Period |Total|Regime|Condition|
+---------+-------+-------+-----+------+---------+
|Andalucía|Almería|2026M03|385.0|NULL  |Nueva    |
|Andalucía|Almería|2026M02|427.0|NULL  |Nueva    |
|Andalucía|Almería|2026M01|421.0|NULL  |Nueva    |
|Andalucía|Almería|2025M12|412.0|NULL  |Nueva    |
|Andalucía|Almería|2025M11|385.0|NULL  |Nueva    |
|Andalucía|Almería|2025M10|538.0|NULL  |Nueva    |
|Andalucía|Almería|2025M09|377.0|NULL  |Nueva    |
|Andalucía|Almería|2025M08|544.0|NULL  |Nueva    |
|Andalucía|Almería|2025M07|429.0|NULL  |Nueva    |
|Andalucía|Almería|2025M06|473.0|NULL  |Nueva    |
|Andalucía|Almería|2025M05|345.0|NULL  |Nueva    |
|Andalucía|Almería|2025M04|369.0|NULL  |Nueva    |
|Andalucía|Almería|2025M03|516.0|NULL  |Nueva    |
|Andalucía|Almería|2025M02|353.0|NULL  |Nueva    |
|Andalucía|Almería|2025M01|400.0|NULL  |Nueva    |
|Andalucía|Almería|2024M12|222.0|NULL  |Nueva    |
|Andalucía|Almería|2024M11|315.

Finally, the date column will be transformed by splitting its information into two separate fields, year and month, in order to facilitate its subsequent analysis.

In [24]:
df = df.withColumn("Year", F.substring(F.col("Period"), 1, 4).cast("int")) \
       .withColumn("Month", F.substring(F.col("Period"), 6, 2).cast("int"))

In [25]:
df = df.drop("Period")

In [26]:
df.show(truncate=False)

+---------+-------+-----+------+---------+----+-----+
|Region   |City   |Total|Regime|Condition|Year|Month|
+---------+-------+-----+------+---------+----+-----+
|Andalucía|Almería|385.0|NULL  |Nueva    |2026|3    |
|Andalucía|Almería|427.0|NULL  |Nueva    |2026|2    |
|Andalucía|Almería|421.0|NULL  |Nueva    |2026|1    |
|Andalucía|Almería|412.0|NULL  |Nueva    |2025|12   |
|Andalucía|Almería|385.0|NULL  |Nueva    |2025|11   |
|Andalucía|Almería|538.0|NULL  |Nueva    |2025|10   |
|Andalucía|Almería|377.0|NULL  |Nueva    |2025|9    |
|Andalucía|Almería|544.0|NULL  |Nueva    |2025|8    |
|Andalucía|Almería|429.0|NULL  |Nueva    |2025|7    |
|Andalucía|Almería|473.0|NULL  |Nueva    |2025|6    |
|Andalucía|Almería|345.0|NULL  |Nueva    |2025|5    |
|Andalucía|Almería|369.0|NULL  |Nueva    |2025|4    |
|Andalucía|Almería|516.0|NULL  |Nueva    |2025|3    |
|Andalucía|Almería|353.0|NULL  |Nueva    |2025|2    |
|Andalucía|Almería|400.0|NULL  |Nueva    |2025|1    |
|Andalucía|Almería|222.0|NUL

In [27]:
df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("../datasets_def/sales_clean")

In [28]:
df.show(truncate=False)

+---------+-------+-----+------+---------+----+-----+
|Region   |City   |Total|Regime|Condition|Year|Month|
+---------+-------+-----+------+---------+----+-----+
|Andalucía|Almería|385.0|NULL  |Nueva    |2026|3    |
|Andalucía|Almería|427.0|NULL  |Nueva    |2026|2    |
|Andalucía|Almería|421.0|NULL  |Nueva    |2026|1    |
|Andalucía|Almería|412.0|NULL  |Nueva    |2025|12   |
|Andalucía|Almería|385.0|NULL  |Nueva    |2025|11   |
|Andalucía|Almería|538.0|NULL  |Nueva    |2025|10   |
|Andalucía|Almería|377.0|NULL  |Nueva    |2025|9    |
|Andalucía|Almería|544.0|NULL  |Nueva    |2025|8    |
|Andalucía|Almería|429.0|NULL  |Nueva    |2025|7    |
|Andalucía|Almería|473.0|NULL  |Nueva    |2025|6    |
|Andalucía|Almería|345.0|NULL  |Nueva    |2025|5    |
|Andalucía|Almería|369.0|NULL  |Nueva    |2025|4    |
|Andalucía|Almería|516.0|NULL  |Nueva    |2025|3    |
|Andalucía|Almería|353.0|NULL  |Nueva    |2025|2    |
|Andalucía|Almería|400.0|NULL  |Nueva    |2025|1    |
|Andalucía|Almería|222.0|NUL

Now, we are going to prepare CSV files to complete the dimensions.

In [31]:
import pandas as pd
df = pd.read_csv("../datasets_def/sales_clean/part-00000-b30d5b13-abb9-4bdf-9a76-d019e3b9fe56-c000.csv", sep=",", header=0)
df_region = df["Region"].drop_duplicates().sort_values()
df_region.to_csv("../datasets_dims/region.csv", index=False)
df_city = df["City"].drop_duplicates().sort_values()
df_city.to_csv("../datasets_dims/city.csv", index=False)